# Agent Registry and Discovery | Agent Infrastructure

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import Dict, List, Optional
from dataclasses import dataclass, field
from datetime import datetime

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Agent Registry and Discovery System

@dataclass
class AgentDescriptor:
    name: str
    description: str
    capabilities: List[str]
    version: str
    endpoint: str  # In production, this would be a URL
    status: str = "active"
    registered_at: str = field(default_factory=lambda: datetime.now().isoformat())
    metadata: Dict = field(default_factory=dict)

class AgentRegistry:
    """Central registry for agent discovery."""
    def __init__(self):
        self._agents: Dict[str, AgentDescriptor] = {}

    def register(self, agent: AgentDescriptor) -> str:
        self._agents[agent.name] = agent
        return f"Registered: {agent.name} v{agent.version}"

    def deregister(self, name: str) -> str:
        if name in self._agents:
            del self._agents[name]
            return f"Deregistered: {name}"
        return f"Agent '{name}' not found"

    def discover(self, capability: str) -> List[AgentDescriptor]:
        """Find all agents with a given capability."""
        return [
            a for a in self._agents.values()
            if capability in a.capabilities and a.status == "active"
        ]

    def discover_best(self, capability: str) -> Optional[AgentDescriptor]:
        """Find the best agent for a capability (highest version)."""
        candidates = self.discover(capability)
        if not candidates:
            return None
        return sorted(candidates, key=lambda a: a.version, reverse=True)[0]

    def list_all(self) -> List[AgentDescriptor]:
        return list(self._agents.values())

    def health_check(self, name: str, healthy: bool):
        if name in self._agents:
            self._agents[name].status = "active" if healthy else "unhealthy"

In [4]:
# Create registry and register agents
registry = AgentRegistry()

registry.register(AgentDescriptor(
    name="financial-analyzer-v2",
    description="Analyzes financial data, generates insights and forecasts",
    capabilities=["financial_analysis", "forecasting", "reporting"],
    version="2.1.0",
    endpoint="http://agents.internal/financial-analyzer",
))

registry.register(AgentDescriptor(
    name="text-summarizer-v1",
    description="Summarizes long documents into key points",
    capabilities=["summarization", "key_extraction"],
    version="1.3.0",
    endpoint="http://agents.internal/summarizer",
))

registry.register(AgentDescriptor(
    name="code-reviewer-v1",
    description="Reviews code for bugs, security issues, and best practices",
    capabilities=["code_review", "security_audit", "best_practices"],
    version="1.0.0",
    endpoint="http://agents.internal/code-reviewer",
))

# Discovery example
print("=== Discovering agents with 'financial_analysis' capability ===")
agents = registry.discover("financial_analysis")
for a in agents:
    print(f"  Found: {a.name} v{a.version} - {a.description}")

print("\n=== Best agent for 'summarization' ===")
best = registry.discover_best("summarization")
if best:
    print(f"  Best: {best.name} v{best.version}")

    # Simulate invoking the discovered agent
    response = model.invoke(
        f"You are '{best.name}': {best.description}.\n\n"
        f"Summarize this: The Federal Reserve kept interest rates unchanged at 5.25-5.50% "
        f"in its latest meeting, citing persistent inflation concerns. Markets reacted positively."
    )
    print(f"  Response: {response.content}")

print("\n=== All registered agents ===")
for a in registry.list_all():
    print(f"  {a.name} (v{a.version}) - capabilities: {a.capabilities} - status: {a.status}")

=== Discovering agents with 'financial_analysis' capability ===
  Found: financial-analyzer-v2 v2.1.0 - Analyzes financial data, generates insights and forecasts

=== Best agent for 'summarization' ===
  Best: text-summarizer-v1 v1.3.0
  Response: The Federal Reserve maintained interest rates at 5.25-5.50% due to ongoing inflation worries, leading to a positive market reaction.

=== All registered agents ===
  financial-analyzer-v2 (v2.1.0) - capabilities: ['financial_analysis', 'forecasting', 'reporting'] - status: active
  text-summarizer-v1 (v1.3.0) - capabilities: ['summarization', 'key_extraction'] - status: active
  code-reviewer-v1 (v1.0.0) - capabilities: ['code_review', 'security_audit', 'best_practices'] - status: active
